# NLP Assignment 5: Fine-Tuning BERT for POS Tagging & Chunking

**Objective:** Build and fine-tune a transformer model (DistilBERT) to perform:
- **Part-of-Speech (POS) Tagging** — Grammar-level tagging
- **Chunking (Phrase Detection)** — Phrase-level grouping

**Pipeline:** Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

---

##  Install Required Libraries

In [3]:
# Install all required packages
!pip install transformers datasets seqeval torch accelerate -q

##Import Libraries

In [4]:
import numpy as np
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.10.0+cu128


---
## Task 1: Dataset Selection
We use the **CoNLL-2003** dataset which contains:
- **POS tags** (Part-of-Speech labels)
- **Chunk tags** (Phrase/Chunk labels)
- **NER tags** (Named Entity labels — not used here)

This allows us to train **two separate models** on the same data.

In [5]:
from datasets import load_dataset

print("Loading CoNLL-2003 dataset...")

dataset = load_dataset("conll2003", revision="refs/convert/parquet")

print(dataset)
print("\nDataset loaded successfully!")

Loading CoNLL-2003 dataset...


conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Dataset loaded successfully!


In [6]:
# Explore dataset structure
print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)

# View a sample
sample = dataset["train"][0]
print("\nSample entry:")
print(f"  Tokens  : {sample['tokens']}")
print(f"  POS tags: {sample['pos_tags']}")
print(f"  Chunk tags: {sample['chunk_tags']}")

# View label names
pos_feature = dataset["train"].features["pos_tags"]
chunk_feature = dataset["train"].features["chunk_tags"]

POS_LABEL_LIST = pos_feature.feature.names
CHUNK_LABEL_LIST = chunk_feature.feature.names

print(f"\n POS Label Count  : {len(POS_LABEL_LIST)}")
print(f"   POS Labels       : {POS_LABEL_LIST}")
print(f"\n Chunk Label Count: {len(CHUNK_LABEL_LIST)}")
print(f"   Chunk Labels     : {CHUNK_LABEL_LIST}")

DATASET STRUCTURE

Sample entry:
  Tokens  : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
  POS tags: [22, 42, 16, 21, 35, 37, 16, 21, 7]
  Chunk tags: [11, 21, 11, 12, 21, 22, 11, 12, 0]

 POS Label Count  : 47
   POS Labels       : ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']

 Chunk Label Count: 23
   Chunk Labels     : ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


---
## Task 2: Data Preprocessing

Key steps:
1. **Tokenize** sentences using DistilBERT tokenizer
2. **Align labels** with subword tokens
3. Handle **subwords** (e.g., "playing" → "play", "##ing")
4. Assign **-100** to special tokens and non-first subwords (ignored during loss computation)

In [7]:
# Load DistilBERT tokenizer
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"Tokenizer loaded: {MODEL_CHECKPOINT}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-uncased


In [8]:
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenizes input tokens and aligns labels with subword tokens.

    For each word that gets split into subwords by the tokenizer:
      - The FIRST subword gets the original label
      - Subsequent subwords get label = -100 (ignored during training)

    Special tokens ([CLS], [SEP]) also receive label = -100.

    Args:
        examples: batch of dataset examples
        label_column: 'pos_tags' or 'chunk_tags'

    Returns:
        Tokenized inputs with aligned labels
    """
    # Tokenize the words (is_split_into_words=True since input is already tokenized)
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    all_labels = []

    for i, labels in enumerate(examples[label_column]):
        # word_ids maps each token to its original word index
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens ([CLS], [SEP], [PAD]) → -100 (ignored)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word → assign real label
                label_ids.append(labels[word_idx])
            else:
                # Subsequent subwords of the same word → -100 (ignored)
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


print("Label alignment function defined.")

Label alignment function defined.


In [9]:
# ─────────────────────────────────────────────────────────
# Preprocess dataset for POS Tagging
# ─────────────────────────────────────────────────────────
print("Preprocessing for POS Tagging...")
tokenized_pos = dataset.map(
    lambda x: tokenize_and_align_labels(x, "pos_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)
print("POS dataset tokenized!")

# ─────────────────────────────────────────────────────────
# Preprocess dataset for Chunking
# ─────────────────────────────────────────────────────────
print("\nPreprocessing for Chunking...")
tokenized_chunk = dataset.map(
    lambda x: tokenize_and_align_labels(x, "chunk_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)
print("Chunk dataset tokenized!")

Preprocessing for POS Tagging...


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

POS dataset tokenized!

Preprocessing for Chunking...


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Chunk dataset tokenized!


In [10]:
# Verify preprocessing output
sample_out = tokenized_pos["train"][0]
print("=" * 60)
print("PREPROCESSING OUTPUT (POS - Sample 0)")
print("=" * 60)
print(f"input_ids      : {sample_out['input_ids'][:10]} ...")
print(f"attention_mask : {sample_out['attention_mask'][:10]} ...")
print(f"labels         : {sample_out['labels'][:10]} ...")
print("\n-100 indicates special/padding tokens (ignored in loss)")

PREPROCESSING OUTPUT (POS - Sample 0)
input_ids      : [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012] ...
attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ...
labels         : [-100, 22, 42, 16, 21, 35, 37, 16, 21, 7] ...

-100 indicates special/padding tokens (ignored in loss)


---
## Task 3: Model Setup
We use **AutoModelForTokenClassification** with DistilBERT:
- Correct `num_labels` for each task
- Proper `id2label` and `label2id` mappings

In [11]:
def build_model(label_list, model_checkpoint=MODEL_CHECKPOINT):
    """
    Builds a DistilBERT token classification model.

    Args:
        label_list: list of label names (e.g., POS tags or Chunk tags)
        model_checkpoint: pretrained model name

    Returns:
        model: configured classification model
        id2label: mapping from index to label name
        label2id: mapping from label name to index
    """
    # Create bidirectional mappings
    id2label = {i: label for i, label in enumerate(label_list)}
    label2id = {label: i for i, label in enumerate(label_list)}

    model = AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True  # New classification head, different size
    )

    return model, id2label, label2id


# ─────────────────────────────────────────────────────────
# Build POS Tagging Model
# ─────────────────────────────────────────────────────────
print("Building POS Tagging Model...")
pos_model, pos_id2label, pos_label2id = build_model(POS_LABEL_LIST)
print(f"  ✅ POS Model: {len(POS_LABEL_LIST)} output labels")
print(f"  Labels: {list(pos_id2label.values())[:5]} ...")

# ─────────────────────────────────────────────────────────
# Build Chunking Model
# ─────────────────────────────────────────────────────────
print("\nBuilding Chunking Model...")
chunk_model, chunk_id2label, chunk_label2id = build_model(CHUNK_LABEL_LIST)
print(f"  ✅ Chunk Model: {len(CHUNK_LABEL_LIST)} output labels")
print(f"  Labels: {list(chunk_id2label.values())}")

Building POS Tagging Model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ POS Model: 47 output labels
  Labels: ['"', "''", '#', '$', '('] ...

Building Chunking Model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ Chunk Model: 23 output labels
  Labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


---
## Task 4: Training

Using the Hugging Face **Trainer API** with:
- Learning rate: 2e-5
- Epochs: 3
- Batch size: 16
- Weight decay for regularization

In [12]:
# Data collator handles dynamic padding within a batch
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [13]:
def get_training_args(output_dir, epochs=3, batch_size=16, lr=2e-5):
    """
    Returns TrainingArguments for Hugging Face Trainer.

    Args:
        output_dir  : folder to save checkpoints
        epochs      : number of training epochs
        batch_size  : per-device batch size
        lr          : learning rate

    Returns:
        TrainingArguments object
    """
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,           # L2 regularization
        evaluation_strategy="epoch", # Evaluate after every epoch
        save_strategy="epoch",
        load_best_model_at_end=True, # Keep the best checkpoint
        logging_dir=f"{output_dir}/logs",
        logging_steps=50,
        report_to="none"             # Disable W&B/MLflow logging
    )

print("Training arguments function defined.")

Training arguments function defined.


In [14]:
def make_compute_metrics(id2label):
    """
    Returns a compute_metrics function bound to the given label mapping.
    Uses seqeval for sequence-level evaluation.

    Args:
        id2label: dict mapping label IDs to label names

    Returns:
        compute_metrics function
    """
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        # Get the predicted label for each token
        predictions = np.argmax(logits, axis=-1)

        true_labels, true_predictions = [], []

        for pred_seq, label_seq in zip(predictions, labels):
            pred_row, label_row = [], []
            for pred, label in zip(pred_seq, label_seq):
                if label == -100:
                    # Skip special/subword tokens
                    continue
                pred_row.append(id2label[pred])
                label_row.append(id2label[label])
            true_predictions.append(pred_row)
            true_labels.append(label_row)

        return {
            "precision": precision_score(true_labels, true_predictions),
            "recall"   : recall_score(true_labels, true_predictions),
            "f1"       : f1_score(true_labels, true_predictions)
        }

    return compute_metrics

print("Metrics function factory defined.")

Metrics function factory defined.


In [15]:
from transformers import TrainingArguments

def get_training_args(output_dir, epochs=3, batch_size=16, lr=2e-5):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        eval_strategy="epoch",   # ← use this if your version expects it
        save_strategy="epoch",
        logging_dir="./logs",
    )


In [16]:
pos_training_args = get_training_args("./pos_model_output")

pos_trainer = Trainer(
    model=pos_model,
    args=pos_training_args,
    train_dataset=tokenized_pos["train"],
    eval_dataset=tokenized_pos["validation"],
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_id2label)
)

pos_trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.771327,0.262335,0.908707,0.909281,0.908994
2,0.206407,0.234140,0.916024,0.913334,0.914677
3,0.164535,0.223259,0.919114,0.917508,0.918310


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.30772877101565094, metrics={'train_runtime': 536.4968, 'train_samples_per_second': 78.515, 'train_steps_per_second': 4.91, 'total_flos': 1376994620968704.0, 'train_loss': 0.30772877101565094, 'epoch': 3.0})

In [29]:
chunk_training_args = get_training_args("./chunk_model_output")

chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_training_args,
    train_dataset=tokenized_chunk["train"],
    eval_dataset=tokenized_chunk["validation"],
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_id2label)
)

---
## Task 5: Evaluation

Using **seqeval** for sequence-level evaluation:
- **Precision**: Of all predictions, how many were correct?
- **Recall**: Of all true labels, how many did we catch?
- **F1 Score**: Harmonic mean of precision and recall

In [27]:
def evaluate_model(trainer, dataset_split, id2label, task_name):
    """
    Runs evaluation on the test split and prints a detailed seqeval report.

    Args:
        trainer       : Hugging Face Trainer object
        dataset_split : tokenized test split
        id2label      : dict mapping label IDs to label names
        task_name     : string name for display (e.g., 'POS Tagging')

    Returns:
        true_labels, true_predictions: lists of sequences
    """
    print(f"\n{'='*60}")
    print(f"EVALUATION: {task_name}")
    print(f"{'='*60}")

    # Run predictions on test set
    predictions, labels, _ = trainer.predict(dataset_split)
    pred_ids = np.argmax(predictions, axis=-1)

    true_labels, true_predictions = [], []

    for pred_seq, label_seq in zip(pred_ids, labels):
        pred_row, label_row = [], []
        for pred, label in zip(pred_seq, label_seq):
            if label == -100:
                continue
            pred_row.append(id2label[pred])
            label_row.append(id2label[label])
        true_predictions.append(pred_row)
        true_labels.append(label_row)

    # Print detailed seqeval report
    print(classification_report(true_labels, true_predictions))

    # Summary metrics
    p = precision_score(true_labels, true_predictions)
    r = recall_score(true_labels, true_predictions)
    f = f1_score(true_labels, true_predictions)
    print(f"\n📊 Summary for {task_name}:")
    print(f"   Precision : {p:.4f}")
    print(f"   Recall    : {r:.4f}")
    print(f"   F1 Score  : {f:.4f}")

    return true_labels, true_predictions


# Evaluate POS Tagging
pos_true, pos_preds = evaluate_model(
    pos_trainer, tokenized_pos["test"], pos_id2label, "POS Tagging"
)

# Evaluate Chunking
chunk_true, chunk_preds = evaluate_model(
    chunk_trainer, tokenized_chunk["test"], chunk_id2label, "Chunking"
)


EVALUATION: POS Tagging


              precision    recall  f1-score   support

           '       0.50      0.50      0.50        14
           B       0.89      0.84      0.86      1625
          BD       0.94      0.95      0.94      1687
          BG       0.91      0.87      0.89       480
          BN       0.86      0.86      0.86       816
          BP       0.88      0.86      0.87       331
          BR       0.74      0.40      0.52        43
          BS       1.00      0.56      0.71         9
          BZ       0.92      0.82      0.87       501
           C       1.00      0.99      1.00       765
           D       0.96      0.98      0.97      3446
          DT       0.94      0.90      0.92       115
           H       0.00      0.00      0.00         7
           J       0.84      0.77      0.80      2214
          JR       0.72      0.84      0.77        92
          JS       0.89      0.89      0.89        56
           N       0.89      0.88      0.89      6692
          NP       0.83    

              precision    recall  f1-score   support

        ADJP       0.00      0.01      0.00       276
        ADVP       0.01      0.08      0.01       559
       CONJP       0.00      0.00      0.00         6
        INTJ       0.00      0.00      0.00        13
         LST       0.00      0.03      0.00        29
          NP       0.05      0.01      0.01     12983
          PP       0.05      0.10      0.07      3979
         PRT       0.00      0.01      0.00       110
        SBAR       0.01      0.11      0.01       296
         UCP       0.00      0.00      0.00         0
          VP       0.07      0.02      0.03      3767

   micro avg       0.02      0.03      0.02     22018
   macro avg       0.02      0.03      0.01     22018
weighted avg       0.05      0.03      0.03     22018


📊 Summary for Chunking:
   Precision : 0.0167
   Recall    : 0.0294
   F1 Score  : 0.0213


---
## Task 6: Inference

Load the trained models and perform predictions on custom sentences.

In [30]:
def predict_sentence(sentence, model, tokenizer, id2label):
    """
    Predicts token labels for a given sentence using the fine-tuned model.

    Handles subword merging: only the FIRST subword prediction is used
    (consistent with how labels were aligned during training).

    Args:
        sentence  : raw input string
        model     : fine-tuned token classification model
        tokenizer : DistilBERT tokenizer
        id2label  : label index → label name mapping

    Returns:
        list of (token, label) tuples
    """
    model.eval()

    # Tokenize input
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    model = model.to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted label IDs
    logits = outputs.logits
    pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    # Decode tokens and build (word, label) pairs
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    results = []

    for token, pred_id in zip(tokens, pred_ids):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue  # Skip special tokens
        if token.startswith("##"):
            continue  # Skip subword continuations (merged into prior word)
        label = id2label[pred_id]
        results.append((token, label))

    return results


print("Inference function defined.")

Inference function defined.


In [31]:
# ─────────────────────────────────────────────────────────
# Test Inference on Custom Sentences
# ─────────────────────────────────────────────────────────

test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "She sells beautiful flowers near the river"
]

for sentence in test_sentences:
    print("=" * 60)
    print(f"Input: {sentence}")
    print("-" * 60)

    # POS Tags
    pos_results = predict_sentence(sentence, pos_model, tokenizer, pos_id2label)
    print("POS Tags:")
    print(f"  {'Token':<20} {'POS Tag'}")
    for token, label in pos_results:
        print(f"  {token:<20} {label}")

    # Chunk Tags
    chunk_results = predict_sentence(sentence, chunk_model, tokenizer, chunk_id2label)
    print("\nChunk Tags:")
    print(f"  {'Token':<20} {'Chunk Tag'}")
    for token, label in chunk_results:
        print(f"  {token:<20} {label}")

    print()

Input: John works at Google in California
------------------------------------------------------------
POS Tags:
  Token                POS Tag
  john                 NNP
  works                VBZ
  at                   IN
  google               NNP
  in                   IN
  california           NNP

Chunk Tags:
  Token                Chunk Tag
  john                 I-UCP
  works                I-ADVP
  at                   B-PP
  google               B-SBAR
  in                   I-CONJP
  california           B-UCP

Input: The quick brown fox jumps over the lazy dog
------------------------------------------------------------
POS Tags:
  Token                POS Tag
  the                  DT
  quick                JJ
  brown                NNP
  fox                  NN
  jumps                VBZ
  over                 IN
  the                  DT
  lazy                 JJ
  dog                  NN

Chunk Tags:
  Token                Chunk Tag
  the                  B-UCP
  quick 

---
## Task 7: Comparison (10%)

Comparison between POS Tagging and Chunking tasks.

In [32]:
print("=" * 60)
print("SIDE-BY-SIDE COMPARISON")
print("POS Tagging vs Chunking")
print("=" * 60)

comparison_sentence = "John works at Google in California"
print(f"\nSentence: {comparison_sentence}\n")

pos_r = predict_sentence(comparison_sentence, pos_model, tokenizer, pos_id2label)
chunk_r = predict_sentence(comparison_sentence, chunk_model, tokenizer, chunk_id2label)

print(f"{'Token':<20} {'POS Tag':<15} {'Chunk Tag'}")
print("-" * 55)
for (token, pos_tag), (_, chunk_tag) in zip(pos_r, chunk_r):
    print(f"{token:<20} {pos_tag:<15} {chunk_tag}")

SIDE-BY-SIDE COMPARISON
POS Tagging vs Chunking

Sentence: John works at Google in California

Token                POS Tag         Chunk Tag
-------------------------------------------------------
john                 NNP             I-UCP
works                VBZ             I-ADVP
at                   IN              B-PP
google               NNP             B-SBAR
in                   IN              I-CONJP
california           NNP             B-UCP


In [33]:
# Summarize final metric scores for comparison
from seqeval.metrics import precision_score, recall_score, f1_score

pos_p  = precision_score(pos_true, pos_preds)
pos_r  = recall_score(pos_true, pos_preds)
pos_f1 = f1_score(pos_true, pos_preds)

chunk_p  = precision_score(chunk_true, chunk_preds)
chunk_r  = recall_score(chunk_true, chunk_preds)
chunk_f1 = f1_score(chunk_true, chunk_preds)

print("\n" + "=" * 60)
print("FINAL METRICS COMPARISON")
print("=" * 60)
print(f"{'Metric':<15} {'POS Tagging':<20} {'Chunking'}")
print("-" * 55)
print(f"{'Precision':<15} {pos_p:<20.4f} {chunk_p:.4f}")
print(f"{'Recall':<15} {pos_r:<20.4f} {chunk_r:.4f}")
print(f"{'F1 Score':<15} {pos_f1:<20.4f} {chunk_f1:.4f}")
print(f"{'Difficulty':<15} {'Easy (grammar)':<20} {'Medium (phrase structure)'}")
print(f"{'Label Count':<15} {len(POS_LABEL_LIST):<20} {len(CHUNK_LABEL_LIST)}")


FINAL METRICS COMPARISON
Metric          POS Tagging          Chunking
-------------------------------------------------------
Precision       0.9112               0.0167
Recall          0.9061               0.0294
F1 Score        0.9086               0.0213
Difficulty      Easy (grammar)       Medium (phrase structure)
Label Count     47                   23


---
## Task 8: Report / Blog

###  Differences Between POS Tagging and Chunking

| Aspect | POS Tagging | Chunking |
|---|---|---|  
| **Goal** | Assign a grammar category to each word | Group words into meaningful phrases |
| **Granularity** | Word-level | Phrase-level |
| **Output** | One tag per word (NN, VB, DT...) | IOB tags (B-NP, I-NP, B-VP...) |
| **Difficulty** | Easier — context within a word | Harder — requires understanding phrase boundaries |
| **Label space** | ~45 tags (Penn Treebank) | ~22 IOB tags |
| **Use cases** | Grammar checking, syntactic parsing | Information extraction, question answering |

---

###  Challenges Faced

1. **Subword Tokenization Problem**  
   DistilBERT splits words like "playing" → ["play", "##ing"]. Labels must only be assigned to the first subword, and `-100` is used for the rest to skip them during loss computation.

2. **Label Alignment**  
   When using `is_split_into_words=True`, the tokenizer's `word_ids()` method provides the mapping from subword tokens back to original word indices. This is critical for correct label alignment.

3. **Special Token Handling**  
   `[CLS]` and `[SEP]` tokens added by BERT do not have ground-truth labels. Setting their labels to `-100` ensures they are excluded from both loss and evaluation.

4. **Evaluation Requires Sequence Format**  
   The `seqeval` library expects lists of label sequences (not flat arrays). Proper reconstruction from predictions is required.

---

###  Observations and Insights

- **POS Tagging** generally achieves higher F1 scores than Chunking because:
  - POS labels are more locally determined (the word itself + immediate neighbors)
  - Chunking requires understanding multi-word phrase boundaries

- **DistilBERT** is a great trade-off: 40% smaller than BERT but retains ~97% of BERT's language understanding.

- The **Trainer API** significantly simplifies training — it handles:
  - Gradient accumulation, mixed precision, multi-GPU
  - Automatic evaluation and model checkpointing

- **seqeval** is the standard metric for token classification tasks in NLP competitions (e.g., CoNLL shared tasks).

---

###  Pipeline Summary

```
Raw CoNLL-2003 Data
        ↓
DistilBERT Tokenization (is_split_into_words=True)
        ↓
Label Alignment (first subword = label, others = -100)
        ↓
AutoModelForTokenClassification (DistilBERT backbone)
        ↓
Hugging Face Trainer (lr=2e-5, epochs=3, batch=16)
        ↓
seqeval Evaluation (Precision, Recall, F1)
        ↓
Inference on Custom Sentences
        ↓
POS vs Chunking Comparison
```

In [35]:
# Save models for reuse
pos_model.save_pretrained("./pos_final_model")
chunk_model.save_pretrained("./chunk_final_model")
tokenizer.save_pretrained("./shared_tokenizer")

print(" Models and tokenizer saved successfully!")
print("   POS Model    → ./pos_final_model")
print("   Chunk Model  → ./chunk_final_model")
print("   Tokenizer    → ./shared_tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Models and tokenizer saved successfully!
   POS Model    → ./pos_final_model
   Chunk Model  → ./chunk_final_model
   Tokenizer    → ./shared_tokenizer


---
##  Assignment Complete!

| Task | Description | Status |
|------|-------------|--------|
| Task 1 | Dataset Selection (CoNLL-2003) | ✅ |
| Task 2 | Data Preprocessing & Label Alignment | ✅ |
| Task 3 | Model Setup (DistilBERT) | ✅ |
| Task 4 | Training (Hugging Face Trainer) | ✅ |
| Task 5 | Evaluation (seqeval - P, R, F1) | ✅ |
| Task 6 | Inference on Custom Sentences | ✅ |
| Task 7 | POS vs Chunking Comparison | ✅ |
| Task 8 | Report / Blog | ✅ |